In [15]:
import os
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.chat_models import init_chat_model
from langchain_community.document_loaders import WebBaseLoader, PyPDFLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from bs4 import SoupStrainer
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
import json
load_dotenv()

True

In [2]:
llm = init_chat_model(model="groq:llama-3.1-8b-instant")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-l6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
pdf = False
url = True
raw_data = False
path = "https://jalammar.github.io/illustrated-transformer/"

In [4]:
if pdf:
    loader = PyPDFLoader("paper.pdf")
    documents = loader.load()
    splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
    splitted = splitter.split_documents(documents)

In [5]:
if url:
    webloader = WebBaseLoader(
        web_path=path,
        bs_kwargs={
            "parse_only": SoupStrainer([
                "main",
                "article",
                "section",
                "h1",
                "h2",
                "h3",
                "h4",
                "p",
                "li"
            ])
        }
    )
    webdocuments = webloader.load()
    splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
    websplits = splitter.split_documents(webdocuments)
websplits

[Document(metadata={'source': 'https://jalammar.github.io/illustrated-transformer/'}, page_content='Jay AlammarVisualizing machine learning one concept at a time.Read our book, Hands-On Large Language Models and follow me on LinkedIn, Bluesky, Substack, X,YouTube \nThe Illustrated Transformer\n\nDiscussions:\nHacker News (65 points, 4 comments), Reddit r/MachineLearning (29 points, 3 comments)\n\n\nTranslations: Arabic, Chinese (Simplified) 1, Chinese (Simplified) 2, French 1, French 2, Italian, Japanese, Korean, Persian, Russian, Spanish 1, Spanish 2, Vietnamese\n\nWatch: MIT’s Deep Learning State of the Art lecture referencing this post\n\nFeatured in courses at Stanford, Harvard, MIT, Princeton, CMU and others'),
 Document(metadata={'source': 'https://jalammar.github.io/illustrated-transformer/'}, page_content="Featured in courses at Stanford, Harvard, MIT, Princeton, CMU and others\n\n\n\n\n\nUpdate: This post has now become a book! Check out LLM-book.com which contains (Chapter 3)

In [6]:
parser = JsonOutputParser()

In [7]:
tones = ["professional", "casual", "bold", "storytelling", "persuasive"]
variation = ["", "Generate a completely different version with a new angle and hook."]

In [8]:
prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a high-quality content repurposing assistant.

STRICT OUTPUT RULES:
- Return ONLY valid JSON
- No explanations or extra text
- Follow schema EXACTLY
- Do NOT rename keys or change structure

SCHEMA:
{{
  "tweets": ["tweet1", "tweet2", "tweet3", "tweet4", "tweet5"],
  "linkedin": "string",
  "email": {{
    "subject": "string",
    "body": "string"
  }}
}}

REQUIREMENTS:
- tweets = exactly 5 strings
- linkedin = single string (NOT array)
- email.subject = string
- email.body = single string with line breaks

----------------------------------------

HARD ENFORCEMENT:

Before returning output, CHECK for these phrases:

- "did you know"
- "most people think"
- "what if i told you"
- "the key is"
- "in today's"
- "want to know"
- "you're not alone"

If ANY appear:
→ Rewrite that part completely
→ Do NOT slightly modify — replace it entirely

----------------------------------------

NO GENERIC / NO FLUFF:

- Avoid obvious statements
- Avoid motivational or filler lines
- Do NOT use:
  "that's when the magic happens"
  "think about it"
  "let that sink in"

----------------------------------------

QUALITY CHECK (MANDATORY):

Before output:
- First line must create curiosity or tension
- Content must NOT sound like teaching
- Content must feel like a real creator insight
- Ending must feel sharp, not soft

If it fails → rewrite internally
"""),

    ("human", """
INSTRUCTIONS:
{instructions}

TONE:
{tone}

{variation_instruction}

CONTENT:
{content}

----------------------------------------

CORE PRINCIPLES:

- Write like exposing a mistake or hidden truth
- Use specific observations, not general advice
- Include at least one non-obvious insight
- Make content feel real, not templated

----------------------------------------

INSIGHT RULE:

Start with a concrete observation.

Bad:
"Content repurposing is important"

Good:
"You publish a blog once… and expect it to work forever"

----------------------------------------

TWEETS:

- Exactly 5 tweets
- Format: "1/5", "2/5", etc.
- First tweet = strongest hook
- Each tweet = standalone insight
- Keep sentences short and sharp
- Avoid questions and engagement bait

----------------------------------------

CONSISTENCY RULE:

- All tweets must match the tone of the first tweet
- If any tweet becomes generic → rewrite it

----------------------------------------

LINKEDIN:

- First line must stop scrolling
- Use short paragraphs (1–2 lines)
- Break lines frequently
- Do NOT explain concepts
- Do NOT use:
  "for example"
  "take X for instance"

----------------------------------------

EMAIL:

- No "Dear [Name]"
- First line must create tension or expose a mistake
- Use short paragraphs with line breaks
- Build curiosity naturally (no bait phrases)
- End with a simple, direct CTA
""")
])

In [9]:
chain = prompt | llm | parser

In [17]:
response = chain.invoke({
    "content": websplits[:4],
    "instructions": """
    1. Create a 5-tweet thread (engaging, numbered, strong hook)
2. Create a LinkedIn post (professional, storytelling style, spaced properly)
3. Create an email newsletter (include subject line, short paragraphs, and CTA)
    """,
    "tone":tones[0],
    "variation_instruction": variation[0]
})

In [19]:
 response

{'tweets': ['1/5 You publish a blog once... and expect it to work forever. Reality check: most content gets stale within months.',
  '2/5 The Illustrated Transformer is not just a blog post, but a gateway to a new era of machine learning. Its impact goes beyond NLP.',
  "3/5 Have you ever wondered why the Transformer outperforms traditional neural networks? It's not just about parallelization, but about a fundamental shift in how we process information.",
  "4/5 The Transformer's success can be attributed to its ability to handle long-range dependencies. But what does this mean for your content strategy?",
  "5/5 Don't just repurpose your content, transform it. The Illustrated Transformer shows us that with the right approach, even old ideas can become new again."],
 'linkedin': "The Illustrated Transformer: A Hidden Truth in Machine Learning\n\nYou publish a blog once... and expect it to work forever. Reality check: most content gets stale within months.\n\n\nThe Transformer, proposed

In [12]:
print(response)

{'tweets': ['1/5 You publish a blog once... and expect it to work forever. Reality: 99% of your audience will miss it the next day.', "2/5 I've seen creators spend 10 hours crafting a single blog post, only to forget about it after a week. Repurpose or perish.", "3/5 You think your blog is enough? Think again. Without consistent short-form content, you're shouting into a void.", "4/5 Repurposing long-form content is not just about efficiency; it's about survival in a world where attention spans are 8 seconds.", "5/5 The most successful creators don't just create content, they create content ecosystems. Don't be a one-hit wonder."], 'linkedin': "You think you're a thought leader, but your content is a one-trick pony. Here's the harsh truth: in today's digital world, consistent short-form content is what separates the champions from the also-rans.\n\nI've seen it happen to the best of them. A single blog post, left to gather dust in the archives. The audience moves on, and so do the resu

In [14]:
types = ["linkedin", "tweets"]
contents = []
for type in types:
    if type in response:
        contents.append(response[type])
contents

[{'subject': 'The Hidden Cost of One-Off Content',
  'body': "You think you're a thought leader, but your content is a one-trick pony. In today's digital world, consistent short-form content is what separates the champions from the also-rans.\n\nI've seen it happen to the best of them. A single blog post, left to gather dust in the archives. The audience moves on, and so do the results.\n\nThe key to success lies not in the quality of your content, but in its versatility. Repurpose your blog posts, turn them into tweets, LinkedIn posts, and email newsletters. Don't let your hard work go to waste.\n\nWant to see real results? Start building a content ecosystem today. Click the link below to learn more."},
 ['1/5 You publish a blog once... and expect it to work forever. Reality: 99% of your audience will miss it the next day.',
  "2/5 I've seen creators spend 10 hours crafting a single blog post, only to forget about it after a week. Repurpose or perish.",
  "3/5 You think your blog is e